# Experimento: [NOME DO EXPERIMENTO]

**Data:** [DATA]  
**Objetivo:** [O QUE ESTOU TESTANDO]  
**Hipótese:** [O QUE EU ESPERO QUE ACONTEÇA]  

---

## 0. Ambiente

In [ ]:
import sys
import os

BRANCH = 'homolog'       # troque para a branch do experimento
REPO   = 'cell-fuzzy-seg'

if not os.path.exists(REPO):
    !git clone --branch {BRANCH} https://github.com/Pedro-io/cell-fuzzy-seg.git

%cd {REPO}
sys.path.append(f"/content/{REPO}")

In [ ]:
!pip install -r requirements.txt

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
from pathlib import Path
from collections import defaultdict

from src.data.load.monuseg_dataset import MonusegDataset
from src.models.networks.marker_net import MarkerNet
from src.losses import (
    LossComposer,
    DiceTerm, SizeTerm, TVTerm, DMapTerm, BorderTerm, RMSETerm
)
from src.io.output_writer import OutputWriter
from src.utils.logger import logger

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Configuração do Experimento

Todos os hiperparâmetros ficam aqui. Se precisar comparar experimentos, copie este bloco e mude só os valores.

In [ ]:
EXPERIMENT_NAME = "exp_001_baseline"

# ── Modelo ────────────────────────────────────────────────────────────────────
MODEL_CONFIG = {
    "encoder_name": "resnet34",  # qualquer encoder do timm/smp
    "pretrained":   True,        # pesos ImageNet no encoder
    "in_channels":  4,           # RGBA: 3 canais da imagem + 1 canal da máscara Cellpose
    "threshold":    0.5,         # limiar para predict() binarizar
}

# ── Treino ────────────────────────────────────────────────────────────────────
TRAIN_CONFIG = {
    "epochs":        30,
    "batch_size":    4,
    "learning_rate": 1e-4,
    "weight_decay":  1e-5,
    "val_split":     0.2,        # fração do training set usada para validação
    "num_workers":   2,
    "seed":          42,
}

# ── Perdas ────────────────────────────────────────────────────────────────────
# Descomente/comente os termos que quiser usar.
# Cada termo tem seus próprios hiperparâmetros — veja src/losses/terms.py.
LOSS_TERMS = [
    DiceTerm(),                  # Dice entre marcadores preditos e gt_masks
    SizeTerm(weight=0.1),        # penaliza diferença de massa total
    TVTerm(weight=0.05, power=1),# suaviza o output espacialmente
    # DMapTerm(weight=0.1),      # penaliza predições longe dos centros
    # BorderTerm(weight=1.0, border_size=50),
]

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR    = Path(f'../results/{EXPERIMENT_NAME}')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Reprodutibilidade
torch.manual_seed(TRAIN_CONFIG['seed'])
np.random.seed(TRAIN_CONFIG['seed'])

print(f'Experimento: {EXPERIMENT_NAME}')
print(f'Output:      {OUTPUT_DIR}')

## 2. Dataset e DataLoader

`MonusegDataset.__getitem__` retorna:
```
{
  'id':           str,
  'image':        np.ndarray  (H, W) ou (H, W, C),  dtype variável
  'ground_truth': np.ndarray  (H, W), binário 0/1
  'meta':         dict
}
```

O DataLoader precisa de uma `collate_fn` customizada porque:
- Imagens podem ter shapes diferentes entre amostras
- Precisamos gerar o `distance_map` a partir do `ground_truth`
- Precisamos montar o tensor de 4 canais que o MarkerNet espera

**Por que 4 canais?** O MarkerNet foi configurado com `in_channels=4`. A ideia é que os 3 primeiros canais sejam a imagem RGB e o 4º seja a máscara de segmentação do Cellpose — assim a rede vê tanto a textura quanto onde o Cellpose achou células.

In [ ]:
TARGET_SIZE = (256, 256)  # redimensiona tudo para o mesmo shape antes de batchificar


def compute_distance_map(binary_mask: np.ndarray) -> np.ndarray:
    """
    Distância euclidiana de cada pixel ao objeto mais próximo.
    Regiões dentro dos objetos têm distância 0; o fundo tem distância positiva.
    Normalizado para [0, 1].
    """
    inverted = 1 - binary_mask.astype(np.float32)
    dmap = distance_transform_edt(inverted).astype(np.float32)
    if dmap.max() > 0:
        dmap /= dmap.max()
    return dmap


def prepare_image(image: np.ndarray, size: tuple) -> np.ndarray:
    """
    Garante que a imagem é (H, W, 3) float32 normalizada em [0, 1].
    """
    import cv2
    if image.ndim == 2:
        image = np.stack([image] * 3, axis=-1)
    elif image.shape[-1] == 1:
        image = np.concatenate([image] * 3, axis=-1)
    image = image[:, :, :3]  # descarta canal extra se existir
    image = cv2.resize(image, size)
    image = image.astype(np.float32)
    if image.max() > 1.0:
        image /= 255.0
    return image


def collate_fn(samples):
    """
    Converte uma lista de dicts do dataset em tensores prontos para o modelo.

    Saída:
        images:        Tensor (N, 4, H, W)  — 3 canais RGB + 1 canal gt_mask
        distance_maps: Tensor (N, 1, H, W)  — distância euclidiana normalizada
        gt_masks:      Tensor (N, 1, H, W)  — máscara binária ground truth
        ids:           List[str]
    """
    import cv2

    images_list, dmaps_list, masks_list, ids = [], [], [], []

    for s in samples:
        img  = prepare_image(s['image'], TARGET_SIZE)            # (H, W, 3) float32
        mask = cv2.resize(s['ground_truth'].astype(np.float32),  # (H, W) float32
                          TARGET_SIZE, interpolation=cv2.INTER_NEAREST)

        # channel 4 = máscara (como proxy da segmentação Cellpose enquanto ela não está no pipeline)
        rgba = np.concatenate([img, mask[:, :, None]], axis=-1)  # (H, W, 4)
        rgba = rgba.transpose(2, 0, 1)                           # (4, H, W)

        dmap = compute_distance_map(mask)                         # (H, W)

        images_list.append(rgba)
        dmaps_list.append(dmap[None])   # adiciona dim de canal: (1, H, W)
        masks_list.append(mask[None])   # (1, H, W)
        ids.append(s['id'])

    return {
        'image':        torch.tensor(np.stack(images_list), dtype=torch.float32),
        'distance_map': torch.tensor(np.stack(dmaps_list),  dtype=torch.float32),
        'ground_truth': torch.tensor(np.stack(masks_list),  dtype=torch.float32),
        'id':           ids,
    }


# ── Dataset ───────────────────────────────────────────────────────────────────
full_dataset = MonusegDataset(
    dataset_name='monuseg',
    config_key='monuseg_training',
)

n_val   = int(len(full_dataset) * TRAIN_CONFIG['val_split'])
n_train = len(full_dataset) - n_val
train_ds, val_ds = random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(TRAIN_CONFIG['seed'])
)

# random_split retorna um Subset — o __getitem__ ainda retorna o dict original

train_loader = DataLoader(
    train_ds,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=True,
    num_workers=TRAIN_CONFIG['num_workers'],
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_ds,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=False,
    num_workers=TRAIN_CONFIG['num_workers'],
    collate_fn=collate_fn,
)

print(f'Train: {n_train} amostras | Val: {n_val} amostras')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

### 2.1 Inspecionar um batch

Antes de treinar, sempre confira shapes e ranges dos tensores.

In [ ]:
sample_batch = next(iter(train_loader))

for key, val in sample_batch.items():
    if isinstance(val, torch.Tensor):
        print(f"{key:>15} | shape {str(val.shape):<25} | min {val.min():.3f} | max {val.max():.3f}")
    else:
        print(f"{key:>15} | {val}")

In [ ]:
# Visualiza uma amostra do batch
idx = 0
img_rgb  = sample_batch['image'][idx, :3].permute(1, 2, 0).numpy()  # (H,W,3)
gt_mask  = sample_batch['ground_truth'][idx, 0].numpy()              # (H,W)
dmap     = sample_batch['distance_map'][idx, 0].numpy()              # (H,W)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_rgb);  axes[0].set_title('Imagem RGB');      axes[0].axis('off')
axes[1].imshow(gt_mask, cmap='gray'); axes[1].set_title('Ground Truth'); axes[1].axis('off')
axes[2].imshow(dmap, cmap='hot');  axes[2].set_title('Distance Map');   axes[2].axis('off')
plt.suptitle(f'id: {sample_batch["id"][idx]}')
plt.tight_layout()
plt.show()

## 3. Modelo

In [ ]:
model = MarkerNet(config=MODEL_CONFIG).to(DEVICE)

# Conta parâmetros treináveis
n_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
print(f'Parâmetros treináveis: {n_params:,}')
print(f'Encoder: {MODEL_CONFIG["encoder_name"]} | in_channels: {MODEL_CONFIG["in_channels"]}')

## 4. Loss e Otimizador

In [ ]:
loss_fn   = LossComposer(terms=LOSS_TERMS).to(DEVICE)
optimizer = optim.Adam(
    model.model.parameters(),
    lr=TRAIN_CONFIG['learning_rate'],
    weight_decay=TRAIN_CONFIG['weight_decay'],
)

# Scheduler opcional — reduz o LR quando a val_loss para de cair
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

print('Loss terms ativos:')
for term in loss_fn.terms:
    print(f'  - {term.name}')

## 5. Loop de Treino

**Fluxo por iteração:**
1. `optimizer.zero_grad()` — zera gradientes acumulados do passo anterior
2. `model.forward(images)` → `markers (N,1,H,W)` — forward pass, constrói grafo
3. `loss_fn(markers, dmaps, gt)` — computa perda total e log por termo
4. `loss.backward()` — percorre o grafo de trás pra frente, calcula ∂loss/∂params
5. `optimizer.step()` — atualiza pesos usando os gradientes

In [ ]:
def run_epoch(loader, training: bool):
    """
    Roda uma epoch completa.
    Retorna: (loss_media, dict_de_termos_medios)
    """
    if training:
        model.model.train()   # ativa BatchNorm/Dropout em modo treino
    else:
        model.model.eval()    # BatchNorm usa estatísticas globais, Dropout desativa

    total_loss   = 0.0
    terms_accum  = defaultdict(float)
    n_batches    = 0

    ctx = torch.no_grad() if not training else torch.enable_grad()

    with ctx:
        for batch in loader:
            images  = batch['image'].to(DEVICE)         # (N, 4, H, W)
            dmaps   = batch['distance_map'].to(DEVICE)  # (N, 1, H, W)
            gt      = batch['ground_truth'].to(DEVICE)  # (N, 1, H, W)

            if training:
                optimizer.zero_grad()

            markers = model.forward(images)              # (N, 1, H, W) ∈ [0,1]
            loss, loss_log = loss_fn(markers, dmaps, gt)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            for name, val in loss_log.items():
                terms_accum[name] += val.item()
            n_batches += 1

    avg_loss  = total_loss / n_batches
    avg_terms = {k: v / n_batches for k, v in terms_accum.items()}
    return avg_loss, avg_terms


print('Pronto para treinar.')

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'terms': defaultdict(list)}
best_val_loss  = float('inf')
best_ckpt_path = CHECKPOINT_DIR / 'best.pt'

for epoch in range(1, TRAIN_CONFIG['epochs'] + 1):
    train_loss, train_terms = run_epoch(train_loader, training=True)
    val_loss,   val_terms   = run_epoch(val_loader,   training=False)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    for name, val in val_terms.items():
        history['terms'][name].append(val)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save(str(best_ckpt_path))
        marker = ' ← best'
    else:
        marker = ''

    terms_str = ' | '.join(f"{k}: {v:.4f}" for k, v in val_terms.items())
    logger.info(f"Epoch {epoch:03d}/{TRAIN_CONFIG['epochs']} | "
                f"train {train_loss:.4f} | val {val_loss:.4f} | "
                f"{terms_str}{marker}")

print(f'\nTreino concluído. Melhor val_loss: {best_val_loss:.4f}')

## 6. Curvas de Aprendizado

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)

n_terms = len(history['terms'])
fig, axes = plt.subplots(1, 1 + n_terms, figsize=(5 * (1 + n_terms), 4))
if n_terms == 0:
    axes = [axes]

axes[0].plot(epochs_range, history['train_loss'], label='train')
axes[0].plot(epochs_range, history['val_loss'],   label='val')
axes[0].set_title('Loss total')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

for ax, (name, vals) in zip(axes[1:], history['terms'].items()):
    ax.plot(epochs_range, vals)
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.grid(True)

plt.suptitle(EXPERIMENT_NAME)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'learning_curves.png', dpi=150)
plt.show()

## 7. Inferência e Visualização

Carrega o melhor checkpoint e gera predições no val set.

In [ ]:
# Carrega o melhor modelo salvo
# model.load() usa torch.load + load_state_dict internamente
model.load(str(best_ckpt_path))
model.model.to(DEVICE)
model.model.eval()

print(f'Checkpoint carregado: {best_ckpt_path}')

In [ ]:
def visualize_predictions(loader, n_samples=4):
    """
    Mostra comparação lado a lado: imagem | ground truth | predição (soft) | predição (binária)
    """
    batch = next(iter(loader))
    images = batch['image'].to(DEVICE)
    gt     = batch['ground_truth'].to(DEVICE)

    with torch.no_grad():
        soft_pred   = model.forward(images)   # (N,1,H,W) ∈ [0,1]
        binary_pred = model.predict(images)   # (N,1,H,W) ∈ {0,1}

    n = min(n_samples, len(images))
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    if n == 1:
        axes = axes[None]

    for i in range(n):
        img_rgb = images[i, :3].permute(1, 2, 0).cpu().numpy()
        axes[i, 0].imshow(img_rgb)
        axes[i, 0].set_title('Imagem')

        axes[i, 1].imshow(gt[i, 0].cpu().numpy(), cmap='gray')
        axes[i, 1].set_title('Ground Truth')

        axes[i, 2].imshow(soft_pred[i, 0].cpu().numpy(), cmap='hot', vmin=0, vmax=1)
        axes[i, 2].set_title('Predição (soft)')

        axes[i, 3].imshow(binary_pred[i, 0].cpu().numpy(), cmap='gray')
        axes[i, 3].set_title(f'Predição (t={model.threshold})')

        for ax in axes[i]:
            ax.axis('off')

    plt.suptitle(f'{EXPERIMENT_NAME} — Val Predictions')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'val_predictions.png', dpi=150)
    plt.show()


visualize_predictions(val_loader, n_samples=4)

## 8. Métricas Finais

Dice e IoU pixel-a-pixel sobre o val set completo.

In [ ]:
def dice_coefficient(pred: torch.Tensor, target: torch.Tensor, eps=1e-7) -> float:
    """
    pred, target: tensores binários qualquer shape
    """
    pred   = pred.float().view(-1)
    target = target.float().view(-1)
    intersection = (pred * target).sum()
    return ((2 * intersection + eps) / (pred.sum() + target.sum() + eps)).item()


def iou(pred: torch.Tensor, target: torch.Tensor, eps=1e-7) -> float:
    pred   = pred.float().view(-1)
    target = target.float().view(-1)
    inter  = (pred * target).sum()
    union  = pred.sum() + target.sum() - inter
    return ((inter + eps) / (union + eps)).item()


all_dice, all_iou = [], []

with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(DEVICE)
        gt     = batch['ground_truth'].to(DEVICE)

        binary_pred = model.predict(images)  # (N,1,H,W)

        for i in range(len(images)):
            all_dice.append(dice_coefficient(binary_pred[i], gt[i]))
            all_iou.append(iou(binary_pred[i], gt[i]))

print(f'Val Dice:  {np.mean(all_dice):.4f} ± {np.std(all_dice):.4f}')
print(f'Val IoU:   {np.mean(all_iou):.4f} ± {np.std(all_iou):.4f}')

## 9. Salvar Outputs Completos

In [ ]:
writer = OutputWriter(output_dir=str(OUTPUT_DIR / 'outputs'))

with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(DEVICE)
        gt     = batch['ground_truth'].cpu().numpy()

        preds = model.predict(images).cpu().numpy()  # (N,1,H,W)

        for i, img_id in enumerate(batch['id']):
            orig_img = images[i, :3].permute(1, 2, 0).cpu().numpy()  # (H,W,3)
            seg      = preds[i, 0]                                     # (H,W)

            writer.save_overlay(
                image_id=img_id,
                image=orig_img,
                segmentation=seg,
                ground_truth=gt[i, 0],
            )

print(f'Outputs salvos em: {OUTPUT_DIR / "outputs"}')

---

## 10. Pipeline de Inferência

O pipeline é exclusivamente para inferência — os modelos chegam já treinados, nenhum gradiente é calculado aqui. Cada step lê do `data` dict, processa, e escreve de volta.

```
image (H,W,C)
  └─► CellposeStep      → data["segmentation"] (H,W) int    — instâncias detectadas
         └─► RGBAStep   → data["rgba"] (H,W,4) uint8        — RGB + alpha da máscara
                └─► MarkerStep [stub]
                       → data["markers"] (H,W) float        — MarkerNet prediz centros
                              └─► SegmentationStep [stub]
                                     → data["segmentation"] (H,W) — segmentação refinada
```

### 10.1 Carregar o modelo treinado

In [ ]:
from src.pipeline.model_pipeline import ModelPipeline
from src.pipeline.steps.cellpose_step import CellposeStep
from src.pipeline.steps.rgba_step import RGBAStep
import cv2

# Troque best_ckpt_path por outro caminho se quiser usar um modelo de experimento diferente
inference_model = MarkerNet(config=MODEL_CONFIG)
inference_model.load(str(best_ckpt_path))
inference_model.model.to(DEVICE)
inference_model.model.eval()

print(f'Modelo carregado: {best_ckpt_path}')

### 10.2 CellposeStep → RGBAStep

Steps implementados. Entrada: `data["image"]`. Saída: `data["segmentation"]` + `data["rgba"]`.

In [ ]:
partial_pipeline = ModelPipeline(steps=[
    CellposeStep(),
    RGBAStep(),
])

sample = full_dataset[0]
data_infer = partial_pipeline.forward(sample, verbose=True)

print('\nKeys após CellposeStep + RGBAStep:')
for k, v in data_infer.items():
    desc = f'shape={v.shape} dtype={v.dtype}' if hasattr(v, 'shape') else type(v).__name__
    print(f'  {k}: {desc}')

### 10.3 MarkerStep — lógica de referência

`MarkerStep` ainda é stub. O código abaixo executa manualmente o que o step precisará fazer quando implementado: pega `data["rgba"]`, pré-processa igual ao treino, roda o modelo, escreve em `data["markers"]`.

In [ ]:
rgba = data_infer['rgba']                                   # (H,W,4) uint8

rgba_resized = cv2.resize(rgba, TARGET_SIZE)                # mesmo size do treino
inp = rgba_resized.astype(np.float32) / 255.0              # normaliza [0,1]
inp = torch.tensor(inp.transpose(2, 0, 1)).unsqueeze(0)    # (1, 4, H, W)
inp = inp.to(DEVICE)

with torch.no_grad():
    markers_tensor = inference_model.predict(inp)           # (1, 1, H, W) binário

data_infer['markers'] = markers_tensor[0, 0].cpu().numpy() # (H, W)

print(f'markers: shape={data_infer["markers"].shape}, valores únicos={np.unique(data_infer["markers"])}')

### 10.4 SegmentationStep — stub

Quando `SegmentationStep` estiver implementado, receberá `data["image"]` + `data["markers"]` e produzirá a segmentação refinada via watershed ou fuzzy segmentation. Por ora, visualizamos o que o pipeline produz até aqui.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(data_infer['image'])
axes[0].set_title('Imagem original')
axes[0].axis('off')

axes[1].imshow(data_infer['segmentation'], cmap='nipy_spectral')
axes[1].set_title('Cellpose (instâncias)')
axes[1].axis('off')

axes[2].imshow(data_infer['rgba'])
axes[2].set_title('RGBA → input MarkerNet')
axes[2].axis('off')

axes[3].imshow(data_infer['markers'], cmap='hot')
axes[3].set_title('Markers (MarkerNet)')
axes[3].axis('off')

plt.suptitle(f'{EXPERIMENT_NAME} — Pipeline | id: {data_infer["id"]}')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'inference_pipeline.png', dpi=150)
plt.show()

### 10.5 Pipeline completo (após implementar os stubs)

Quando `MarkerStep` e `SegmentationStep` estiverem prontos, todo o bloco 10.2–10.4 vira isto:

```python
full_pipeline = ModelPipeline(steps=[
    CellposeStep(),
    RGBAStep(),
    MarkerStep(model=inference_model, target_size=TARGET_SIZE, device=DEVICE),
    SegmentationStep(),
])

writer = OutputWriter(str(OUTPUT_DIR / 'outputs'))
for i in range(len(full_dataset)):
    result = full_pipeline.forward(full_dataset[i])
    writer.save_all(result)
```

---

## 11. Resumo

Preencha após rodar o experimento.

| Métrica  | Valor |
|----------|-------|
| Val Loss |       |
| Val Dice |       |
| Val IoU  |       |
| Epochs   |       |

**O que funcionou:**

**O que não funcionou:**

**Próximo experimento:**